In [2]:
#Install Libraries

!pip install transformers torch scikit-learn pandas

In [8]:
#Extract dataset
import zipfile

zip_path = "archive (1).zip"
extract_path = "imdb_data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction Done ✅")


Extraction Done ✅


In [9]:
#Load Dataset
import pandas as pd

train_df = pd.read_csv("imdb_data/Train.csv")
val_df   = pd.read_csv("imdb_data/Valid.csv")
test_df  = pd.read_csv("imdb_data/Test.csv")

print(train_df.head())

                                                text  label
0  I grew up (b. 1965) watching and loving the Th...      0
1  When I put this movie in my DVD player, and sa...      0
2  Why do people who do not know what a particula...      0
3  Even though I have great interest in Biblical ...      0
4  Im a die hard Dads Army fan and nothing will e...      1


In [10]:
print(train_df.columns)

Index(['text', 'label'], dtype='object')


In [11]:
train_df.rename(columns={'review':'text', 'sentiment':'label'}, inplace=True)
val_df.rename(columns={'review':'text', 'sentiment':'label'}, inplace=True)
test_df.rename(columns={'review':'text', 'sentiment':'label'}, inplace=True)

In [12]:
#Preprocessing
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df['text'] = train_df['text'].apply(clean_text)
val_df['text']   = val_df['text'].apply(clean_text)
test_df['text']  = test_df['text'].apply(clean_text)

In [13]:
train_df = train_df.dropna(subset=['text', 'label'])
val_df   = val_df.dropna(subset=['text', 'label'])
test_df  = test_df.dropna(subset=['text', 'label'])

In [14]:
train_df['label'] = train_df['label'].astype(int)
val_df['label']   = val_df['label'].astype(int)
test_df['label']  = test_df['label'].astype(int)

In [15]:
#Tokenization
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

train_encodings = tokenizer(list(train_df['text']), truncation=True, padding=True, max_length=128)
val_encodings   = tokenizer(list(val_df['text']), truncation=True, padding=True, max_length=128)
test_encodings  = tokenizer(list(test_df['text']), truncation=True, padding=True, max_length=128)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [16]:
#Dataset class
import torch
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)  # IMPORTANT
        return item

    def __len__(self):
        return len(self.labels)

In [17]:

#Create DataSet + DataLoader
from torch.utils.data import DataLoader

train_dataset = CustomDataset(train_encodings, train_df['label'])
val_dataset   = CustomDataset(val_encodings, val_df['label'])
test_dataset  = CustomDataset(test_encodings, test_df['label'])

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16)
test_loader  = DataLoader(test_dataset, batch_size=16)

In [18]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [19]:
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

In [20]:
import torch.nn as nn

def train(model, loader):
    model.train()
    total_loss = 0

    loss_fn = nn.CrossEntropyLoss()

    for batch in loader:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        loss = loss_fn(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [21]:

#Evaluation function
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

def evaluate(model, loader):
    model.eval()
    predictions, true_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)

            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(true_labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average='binary')
    cm = confusion_matrix(true_labels, predictions)

    return acc, precision, recall, f1, cm

In [ ]:
#Train Model
for epoch in range(3):
    print(f"\n🚀 Epoch {epoch+1}/3")

    train_loss = train(model, train_loader)
    acc, prec, rec, f1, cm = evaluate(model, val_loader)

    print(f"Loss: {train_loss:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")


🚀 Epoch 1/3


In [ ]:
#final testing
acc, prec, rec, f1, cm = evaluate(model, test_loader)

print("\n🎯 Final Results")
print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1 Score:", f1)
print("Confusion Matrix:\n", cm)